# Notebook 2 <br>Cleaning, transforming, summarizing

In the last notebook we looked at how we can inspect data using Pandas. Now we will see how you can also modifiy data.

Start by loading Pandas:

In [ ]:
import pandas as pd
import numpy as np
import pooch

%reload_ext autoreload
%autoreload 2

pd.set_option("display.precision", 2)

This notebook needs a few extra files: the data you will work with, and `tests.py`, which checks your answers. Run the cell below to download them into the folder this notebook is in. You only need to do this once; after that the files are already on your computer and the cell finishes immediately.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/spcourse/pandas/main/pandas-dp/data/"

print("downloading data files")
for fname in ["tracks_2024.csv", "festivals.csv", "tests.py"]:
    pooch.retrieve(url=DATA_URL + fname, known_hash=None, fname=fname, path=".", progressbar=True)
print("done!")

from tests import *

## Chapter 5: types & parsing

In Notebook 1 you learned to diagnose a messy file: you loaded the raw seabird export and ran some sanity checks. Finding the problems is the first half of the job. Of course, once you know what the problems are, you will want to repair them. So in this chapter we'll look at some basic ways you can do this.

We'll look at: `read_csv`'s parameters, `to_numeric`, `astype`, `to_datetime`, and the `.str` accessor for tidying up text.

### The data

`tracks_2024.csv` is an export from a music-streaming platform: one row per track, with how often it was played in the last 30 days.

> Disclaimer! **This is a synthetic dataset, generated using AI.** The platform does not even exist and every artist, track and number in the file was generated for teaching purposes only.

#### Data dictionary (codebook)

| column | type | expected values |
|---|---|---|
| `track_id` | integer | unique identifier |
| `released_at` | text (YYYY-MM-DD) | release date, somewhere in 2015–2024 |
| `artist` | text | one of 12 artists |
| `genre` | text | one of: `Classical`, `Electronic`, `Hip-Hop`, `Jazz`, `Pop`, `Rock` |
| `streams` | whole number | plays in the last 30 days |
| `avg_rating` | number | average listener rating, 1.0–5.0 |
| `playlist_adds` | whole number | how often the track was added to a playlist |
| `explicit` | text | `yes` or `no` |

The export is documented as containing **500 tracks**.

### Read the data

In Notebook 1 we said that `read_csv` has a lot of parameters you can configure, but that for that particular file we didn't need any of them — the defaults worked fine.

This time they don't. Let's see what happens:

In [ ]:
raw = pd.read_csv("tracks_2024.csv")

Pandas could not read the file, you should get:

> `ParserError: Error tokenizing data. C error: Expected 1 fields in line 6, saw 8`

This is because the file is not completely formatted as a .csv.

Have a look at the first few lines:

In [ ]:
with open("tracks_2024.csv") as file:
    for line_number, line in list(enumerate(file, start=1))[:10]:
        print(line_number, "|", line.rstrip())

Before the actual data there are five lines of export metadata. And these lines cannot be parsed as csv.

Files like this are common. Anything exported by a measurement device, a database tool or a reporting dashboard tends to carry a header block like this. Line 6 is the real header, and the data starts on line 7. Everything above is preamble. So we tell `read_csv` to ignore the first five lines with `skiprows`:

In [ ]:
raw = pd.read_csv("tracks_2024.csv", skiprows=5)

print("shape:", raw.shape)
display(raw.head())

500 rows and 8 columns, which matches what the codebook promised. The column names are right, and the values line up under them.

This works, but it isn't very robust: if the export ever gains or loses a preamble line, `skiprows=5` silently starts cutting into the data.

A neater solution: Since every preamble line here begins with a `#`, we can instead tell pandas to treat `#` as the start of a comment and ignore those lines wherever they appear:

In [ ]:
alternative = pd.read_csv("tracks_2024.csv", comment="#")

print("shape:", alternative.shape)
display(alternative.head())

Both work. Which one you reach for depends on the file: `comment="#"` is more robust when the preamble is marked as comments, `skiprows` is what you need when it isn't (which is common — plenty of exports just put a few bare lines of text at the top).

### More `read_csv` parameters

`skiprows` and `comment` are two of a long list. You don't need to memorise it, but it helps to know which kinds of problems `read_csv` can solve for you at load time. Here are some important parameters:

| parameter | what it does |
|---|---|
| `parse_dates` | read one or more columns as dates instead of text (`parse_dates=["released_at"]`) |
| `na_values` | extra values that should count as missing, such as a `-999` sentinel |
| `skipinitialspace` | drop the space after each separator, so `a, b` doesn't become `" b"` |
| `sep` | the separator, if it isn't a comma (`sep=";"`, `sep="\t"`) |
| `decimal` | the decimal marker, if it isn't a point (`decimal=","`) |
| `thousands` | the thousands separator, so `"1,061"` reads as `1061` |
| `index_col` | which column to use as the index |
| `usecols` | read only some columns, instead of all of them |
| `nrows` | read only the first *n* rows — useful for peeking at a huge file |
| `encoding` | for files that aren't UTF-8 (`encoding="latin-1"`) |

Keep `decimal` in mind: `avg_rating` in `tracks_2024.csv` uses a comma. The full list is in the [`read_csv` documentation](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html), which is worth a skim once so you know what's on it.


### Exercise 12

For this exercise we'll use a very small data file: `festivals.csv`. Have a look at the content first:

In [ ]:
with open("festivals.csv") as file:
    print(file.read())

Read this file into a DataFrame called `festivals`, using the right parameters from the table above.

Be careful: with the default settings `read_csv` does **not** raise an error on this file. It happily returns something — just not anything useful. That is the more dangerous kind of failure, and the reason the checks below matter.

Your result should have:

- **6 rows and 4 columns**
- columns named exactly `festival`, `city`, `headline_artist` and `attendance` — with no stray spaces, so that `festivals["city"]` works
- `attendance` as a number, not text

In [ ]:
festivals = None

# YOUR CODE HERE

display(festivals)
print("shape:  ", festivals.shape)
print("columns:", list(festivals.columns))
print()
print(festivals.dtypes)

In [ ]:
test_12(festivals)

### Validation

Now let's go back to the `tracks_2024.csv` file. It was loaded in the `raw` DataFrame, but that is not the same as the data being usable. So let's inspect it further with `info()`:

In [ ]:
raw.info()

There are some issues here. First of all, the dtypes are wrong. Go back to the codebook: `released_at`, `streams`, `avg_rating` and `playlist_adds` all have the wrong type:

| column | codebook description | actual dtype |
|---|---|---|
| `released_at` | a date | `str` |
| `streams` | a whole number | `str` |
| `avg_rating` | a number | `str` |
| `playlist_adds` | a whole number | `float64` |

A column that should be numeric arriving as `str` means pandas found something in it that isn't a number. Until that is fixed, `raw["streams"].mean()` will not give you an average — it will either fail or do something worse.

`playlist_adds` is the odd one out: it *is* numeric, just not the whole number the codebook promised. This is because some of its values are missing. The `int64` type has no way to represent "no value", but with `float64` you can use `NaN` (not a number) for missing values. So the moment a column of whole numbers contains a single missing value, pandas has to interpret the entire column as `float64` in order to represent it correctly.

This connects to a second issue: some values are missing. The non-null counts aren't all 500, meaning that some columns contain missing values.

The column `streams` is even stranger. Here a whole number is represented as a string. That is not because of missing values, as Pandas would have used `float64` if that were the only issue. Something else is wrong in that column. We will find out what soon.

### Converting text to numbers: `to_numeric`

So what is wrong with `streams`? Let's try to convert it to a number and see what happens.

`pd.to_numeric()` converts a Series to a numeric dtype. Its `errors` parameter decides what happens when it meets a value it cannot parse:

- `errors="raise"`: (the default) stop and raise an error
- `errors="coerce"`: give up on that value and replace it with `NaN`. Keep in mind that this does delete whatever it did not parse as a number.

In [ ]:
streams_numeric = pd.to_numeric(raw["streams"], errors="coerce")

print("missing before:", raw["streams"].isna().sum())
print("missing after: ", streams_numeric.isna().sum())

The column started with 11 missing values and now has 129. The conversion silently destroyed 118 values, and pandas did not warn us about a single one of them.

Before accepting that, let's look at what it threw away:

In [ ]:
could_not_parse = raw["streams"][streams_numeric.isna() & raw["streams"].notna()]

print(len(could_not_parse), "values could not be parsed")
display(could_not_parse.head(15))

These all look like numbers, just written down differently. `498K` is probably `498000` and `ca. 85000` should maybe just be `85000`. These numbers were all thrown away by `errors="coerce"`. So, it is not a repair but a decision to throw data away. It's up to you to decide whether that is acceptable for your use case.

### Recovering the abbreviated numbers with `.str`

Some of these values can be recovered: The `K`/`M` ones follow an easy rule: `498K` means `498 × 1000` and `1.3M` means `1.3 × 1000000`. We can write code to fix this.

Others (`ca. 85000`, `~34000`, `1100+`, `about 9300`) follow no clear rule. You would need a special case for each, and even then `~34000` is probably somebody's guess rather than a measurement. We'll leave those alone for now.

To pull the abbreviations apart we need string operations on a whole column at once. That's what `.str` is for: it applies Python's string methods elementwise.

```
column.str.lower()             # every value lowercased
column.str.strip()             # whitespace removed from both ends
column.str.endswith("K")       # a boolean Series: does each value end in "K"?
column.str.rstrip("KM")        # remove a trailing "K" or "M" if there is one
```

### Exercise 13

Convert `raw["streams"]` into a proper numeric Series called `streams`, recovering the abbreviated values along the way:

- a value ending in `K` is that number times **1 000**
- a value ending in `M` is that number times **1 000 000**
- anything else that isn't a number stays missing

Then print how many values are still missing afterwards.

> **Hint.** The column contains `NaN`, and `.str.endswith("K")` returns `NaN` for those rather than `False`, which breaks boolean indexing. Pass `na=False` to get a usable mask: `.str.endswith("K", na=False)`.

In [ ]:
streams = None

# YOUR CODE HERE

print("still missing:", streams.isna().sum())

In [ ]:
test_13(streams)

20 missing, down from 129. So we got 109 values back. Compare the column before and after the repair:

In [ ]:
print("            coerce only   after repair")
print(f"mean    {streams_numeric.mean():>12,.0f}   {streams.mean():>12,.0f}")
print(f"max     {streams_numeric.max():>12,.0f}   {streams.max():>12,.0f}")
print(f"count   {streams_numeric.count():>12,.0f}   {streams.count():>12,.0f}")

One reason why it was very important to fix these entries is that discarded values were not at all random. They were all values above `100000`. So just dropping them can be dramatic for any further analysis. The mean without conversion was **24 000**, after it became **120 000**. So just dropping the badly parsed values made "average track" wrong by a factor of five. 

So an important takeaway here is that whenever a conversion changes the number of missing values, find out which values went and why!

The 20 that are still missing are the 9 hand-typed estimates plus 11 rows that were empty in the file to begin with. We wil deal with this in Chapter 6.

### Converting with `astype`

`astype` also converts a column, but unlike `to_numeric` it has no `errors="coerce"`. It either converts everything, or it fails. Try it on `avg_rating`:

In [ ]:
raw["avg_rating"].astype(float)

This produces and error!

> `ValueError: could not convert string to float: '3,5'`

The ratings use a decimal comma instead of a point (which is normal in many countries). Replace the comma and the conversion works:

In [ ]:
avg_rating = raw["avg_rating"].str.replace(",", ".").astype(float)

print(avg_rating.dtype, f"{avg_rating.min()} - {avg_rating.max()}", "| missing:", avg_rating.isna().sum())

Or you could have avoided it entirely by telling `read_csv` about it while loading:

```python
pd.read_csv("tracks_2024.csv", skiprows=5, decimal=",")
```

Both give the same result. Fixing it at load time is tidier when the whole column uses one consistent separator; `.str.replace` is what you need when the mess is more irregular than a single parameter can describe.

Note that `astype` is happy here even though 23 values are missing — `NaN` is a valid `float`. What it will not tolerate is text it cannot parse. Use `astype` when you expect the data to be clean and want to hear about it if it isn't, and `to_numeric(errors="coerce")` when you know it isn't and want to carry on anyway.

### Tidying up text

The codebook says there are 12 artists. Let's check:

In [ ]:
print("distinct artists:", raw["artist"].nunique())
display(raw["artist"].value_counts())

So, 18 values instead of 12, because in some cases the same artist has been written in several ways.

`explicit` has the same problem:

In [ ]:
display(raw["explicit"].value_counts(dropna=False))

Seven spellings of what should be two values, plus 15 missing.

### Exercise 14

Clean up both columns. Create:
1. a Series called `artist` with exactly **12** distinct values.
2. a Series called `explicit` containing only `"yes"`, `"no"` and `NaN`.

Useful methods: `.str.strip()`, `.str.lower()`, `.str.title()`, `.str[0]` (the first character of each value), and `.replace({...})` to swap values using a dictionary.

> **Watch out.** Stripping and re-capitalising gets you *close*, but not all the way. Check the count — if you get 13, look carefully at what is left over.

In [ ]:
artist = None
explicit = None

# YOUR CODE HERE

display(artist.value_counts())
display(explicit.value_counts(dropna=False))

In [ ]:
test_14(artist, explicit)

### Dates: `to_datetime`

`released_at` is still text. You can use `pd.to_datetime()` to convert it.

In [ ]:
released_at = pd.to_datetime(raw["released_at"])

print(released_at.dtype, "| missing:", released_at.isna().sum())
print("range:", released_at.min().date(), "to", released_at.max().date())
display(released_at.dt.year.value_counts().sort_index())

2015 to 2024, which is what the codebook promised, and every year has a plausible number of releases. Converting a date column is one of the easiest things to get quietly wrong (an American `03/04` and a European `03/04` are different days). So, always check the range afterwards.

### Putting it back together

We have been repairing columns one at a time, each into its own Series. Collect them into one DataFrame so the next chapters have something to work with:

In [ ]:
tracks = raw.copy()

tracks["released_at"] = released_at
tracks["artist"] = artist
tracks["streams"] = streams
tracks["avg_rating"] = avg_rating
tracks["explicit"] = explicit

tracks.info()

Compare that with the table at the start of the chapter. `released_at` is a date, `artist` and `explicit` are tidy text, and the two numeric columns are numeric.

`streams` and `playlist_adds` are `float64` rather than whole numbers for the reason we saw earlier: both still contain missing values, and `int64` cannot hold those. They will become whole numbers once the missing values are dealt withmin the next chapter.

Which is all that is left:

In [ ]:
display(tracks.isna().sum())

None of these are parsing problems any more, and none of them have a single right answer. Whether you drop those rows, fill them in, or leave them alone depends on what the column means and what you intend to do with it.

## Chapter 6: Missing data

Chapter 5 fixed the *types*. What it could not fix is the values that simply aren't there.

`NaN` looks like one thing, but it is not. A missing `playlist_adds` and a missing `streams` are missing for completely different reasons, and they need completely different treatment. Deciding which is which is the part that needs you; the pandas is easy.

### Finding it

`isna()` gives a DataFrame of booleans. Summing it counts the gaps per column; taking the mean gives the proportion, which is usually the more useful number:

In [ ]:
print("missing values:")
print(tracks.isna().sum())

print("\nas a percentage:")
print((tracks.isna().mean() * 100).round(1))

First we have to understand what each gap means.

| column | missing | what it means | reasonable response |
|---|---|---|---|
| `playlist_adds` | 90 (18%) | the track was never added to a playlist | this shold be filled with 0 |
| `avg_rating` | 23 (4.6%) | nobody rated the track | no rating exists, we could choose to fill or leave it |
| `streams` | 20 (4%) | the value was unreadable in the export | we have no idea what it was so we should drop it |
| `genre` | 40 (8%) | not recorded | leave (for now) |
| `explicit` | 15 (3%) | not recorded | leave |

Only one of those is a data-cleaning question. The rest are questions about what the numbers mean, and no library can answer them for you.

### Dropping rows: `dropna`

`dropna()` removes rows containing missing values. By default it drops a row if any column is missing (which is almost never what you want):

In [ ]:
print(f"original:                   {len(tracks)}")
print(f"dropna():                   {len(tracks.dropna())}")
print(f"dropna(subset=['streams']): {len(tracks.dropna(subset=['streams']))}")

The naive version throws away 188 rows (more than a third of the dataset) because it also drops every track with no genre, no rating, or no playlist adds. Those rows were perfectly usable.

`subset` restricts the check to the columns you actually care about. Other useful arguments:

- `axis=1` drops columns instead of rows
- `how="all"` only drops when the whole row (or column) is missing
- `thresh=n` keeps rows that have at least *n* real values

### Filling: `fillna`

`fillna` replaces missing values instead of removing the row. You can pass a single value, or a dictionary to use a different value per column:

```python
df.fillna(0)                          # everything gets 0
df.fillna({"playlist_adds": 0, "avg_rating": 3.5})
df["x"].fillna(df["x"].mean())        # the column's own average
```

There is also `ffill()` and `bfill()`, which carry the previous or next value forward. Those only make sense when the rows are in a meaningful order.

### Exercise 15

Build a cleaned-up DataFrame called `clean`, starting from `tracks`:

1. drop the rows where `streams` is missing (and only those!)
2. fill `playlist_adds` with `0`, and make the column a whole number (`int`)
3. fill `avg_rating` with the average rating
4. leave `genre` and `explicit` alone

In [ ]:
clean = None

# YOUR CODE HERE

print("rows:", len(tracks), "->", len(clean))
print()
print(clean.isna().sum())

In [ ]:
test_15(clean)

### Validation

Two things are worth checking after any missing-data step.
- We dropped 20 rows and expected to drop 20.
- How filling changed the data. Filling with the mean is convenient, but it affects the data:

In [ ]:
before = tracks.dropna(subset=["streams"])["avg_rating"]
after = clean["avg_rating"]

print(f"          before    after")
print(f"count  {before.count():>8}  {after.count():>8}")
print(f"mean   {before.mean():>8.3f}  {after.mean():>8.3f}")
print(f"std    {before.std():>8.3f}  {after.std():>8.3f}")

The mean is identical, but the standard deviation dropped. We invented 23 tracks with exactly average ratings, so the data now looks more consistent than it really is. Something you need to be aware of if you're going to do further analysis.

*Further reading:* the Python Data Science Handbook on [handling missing data](https://jakevdp.github.io/PythonDataScienceHandbook/03.04-missing-values.html).

## Chapter 7: Deriving & transforming columns

Pandas also allows us to transform data beyond the simple repairs we saw before. In this chapter we look at three ways to do that: arithmetic on whole columns at once, applying your own function with `map`, and turning numbers into categories with `np.where` and `cut`.

### Operating on DataFrames

In the section on [`Adding and modifying data`](#Adding-and-modifying-data) you already saw that basic arithmetic operations are actually done *element-wise* by default, which is extremely useful. The general term for operations that are applied element-wise are "Universal Functions" (or Ufuncs) and make libraries such as NumPy and Pandas incredibly powerful, convenient and computationally efficient. These Ufuncs do not only work with basic arithmetics (`+`, `-`, `*`, `/`, etc) but also more sophisticated operations (exponential, trigonometric, logarithmic functions and many more).

With all of these operations, it is important to keep track of the `index` of the Series you are using. For example, if you have a Series `a` with index `0,1,2` and a Series `b` with index `1,2,3`, the operation `c = a + b` will have index `0,1,2,3` (the union of the index of `a` and `b`). Only on indices `1` and `2` will it be able to compute a result, and on indices `0` and `3` it will automatically fill in `NaN` ("Not a Number").

In [ ]:
a = pd.Series(
    data=[5, 2, 3],
    index=[0, 1, 2]
)
b = pd.Series(
    data=[7, 3, 1],
    index=[1, 2, 3]
)

print("Series a:")
display(a)

print("\n\nSeries b:")
display(b)

print("\n\na+b gives:")
display(a + b)

As you can see, the indices `1` and `2` have been added element-wise, as they occur in *both* Series, while for indices `0` and `3` no valid result could be computed, so they become `NaN`. If you want a different result for the indices that only occur once, you'll need to specify how to handle those cases with the `pd.Series.add(...)` method. Using `a.add( b )` is completely equivalent to `a + b`, but using `.add` allows you to specify a `fill_value`, which it will use whenever one of the Series is missing a value for some index.

For example, suppose the platform recorded a few playlist additions late, and sends them to you as a separate Series covering only those tracks. Adding it directly would leave every *other* track as `NaN`. Using `.add(...)` with `fill_value=0` treats the missing entries as zero instead.

In [ ]:
# index by track_id so the two Series can be matched up per track
adds = clean.set_index("track_id")["playlist_adds"]

late = pd.Series(
    data=[12, 5, 30],
    index=[10001, 10004, 10009],
    name="late_adds"
)

print("plain addition:")
display((adds + late).head(4))

print("with fill_value=0:")
display(adds.add(late, fill_value=0).head(4))

print("NaN produced —  plain:", (adds + late).isna().sum(),
      " with fill_value:", adds.add(late, fill_value=0).isna().sum())

### Exercise 16

The platform re-counted a handful of tracks and found streams it had missed. The corrections are in the Series below, indexed by `track_id`.

Add them to the stream counts, and store the result in `corrected` (indexed by `track_id`).

Every track that is *not* in the correction list must keep its original value — not become `NaN`.

In [ ]:
streams_by_id = clean.set_index("track_id")["streams"]

missed = pd.Series(
    data=[25_000, 1_200, 340_000, 8_500],
    index=[10002, 10015, 10061, 10112],
    name="missed_streams"
)

corrected = None

# YOUR CODE HERE

print("tracks:", len(corrected), " missing:", corrected.isna().sum())
display(corrected.loc[[10002, 10015, 10061, 10112]])

In [ ]:
test_16(corrected)

*Further Reading:* The python data science handbook on [operations in pandas](https://jakevdp.github.io/PythonDataScienceHandbook/03.03-operations-in-pandas.html).

### Map

Next to these built-in operations, you can also apply your own function to each value in a Series. Below is a very simple example function, that just returns the length of a string. Say you want to apply `myFunc` to each value in `series_in` and save the results in `series_out`, then you might write something like this:

In [ ]:

# A very simple example function
def myFunc(value):
    newValue = len(value)
    return newValue


# Get some Series
series_in = pd.Series(["Hello", "world!"])

print("Input series:")
display(series_in)


# Save the new results in a list for now
series_out = []

# Go through each value in the input Series
for value in series_in:
    # Apply your function to the value, save in output list
    series_out.append( myFunc(value) )

# Create a new series with new values
series_out = pd.Series(series_out)


print("\n\nOutput Series:")
display(series_out)

But looping through values, appending them and then creating a new Series is quite inefficient. All of this can be replaced by using the `pd.Series.map` method. You provide the name of your function (`myFunc` in this case), and Pandas will *map* each value in your Series using your function:

In [ ]:

# A very simple example function
def myFunc(value):
    # Do some operations here...
    newValue = len(value)
    return newValue


# Get some Series
series_in = pd.Series(["Hello", "world!"])

print("Input series:")
display(series_in)


# Apply myFunc to each value in series_in, save the result in series_out
series_out = series_in.map(myFunc)


print("\n\nOutput Series:")
display(series_out)

### Exercise 17

In Chapter 5 you turned the platform's `498K` notation into a number. Now go the other way: for a report, the numbers need to be written in that short form again.

Use `map` and the `format_streams` function below to create a new column `clean["streams_short"]`.

In [ ]:
def format_streams(n):
    """
    Write a stream count the way the platform does.

    Input:  n - a number of streams
    Output: a string, e.g. 1300000 -> '1.3M', 498000 -> '498K', 3439 -> '3439'
    """
    if n >= 1_000_000:
        return f"{n / 1_000_000:.1f}M"
    if n >= 100_000:
        return f"{round(n / 1_000)}K"
    return str(int(n))


# YOUR CODE HERE

display(clean[["streams", "streams_short"]].head())

In [ ]:
test_17(clean)

### Conditional and binned columns

Two more ways to make a column, both turning numbers into categories.

`np.where(condition, if_true, if_false)` picks between two values row by row. `pd.cut` splits a numeric column into labelled categories.

In [ ]:
# a part of the date
clean["year"] = clean["released_at"].dt.year

# a yes/no flag — this is exactly the rule the export used when it wrote "498K"
clean["was_abbreviated"] = np.where(clean["streams"] >= 100_000, "yes", "no")

# a set of bands
clean["popularity"] = pd.cut(
    clean["streams"],
    bins=[0, 10_000, 100_000, 1_000_000, np.inf],
    labels=["niche", "modest", "popular", "hit"]
)

display(clean[["streams", "year", "was_abbreviated", "popularity"]].head())

### Validation

A derived column is new data, so it gets the same treatment as data you read from a file. Look at its range if it's a number, and its `value_counts()` if it's a category:

In [ ]:
print("year:", clean["year"].min(), "-", clean["year"].max())
print()
print(clean["was_abbreviated"].value_counts())
print()
print(clean["popularity"].value_counts().sort_index())
print()
print("unexpected NaN:", clean[["year", "was_abbreviated", "popularity"]].isna().sum().sum())

- Years run 2015–2024, which matches the codebook
- The 109 abbreviated tracks are exactly the ones `to_numeric` we initially destroyed back in Chapter 5.
- `cut` did not silently create any `NaN`s. (If you had written `bins=[1000, 10_000, 100_000]` every track outside that range would have quietly vanished from the column.)

## Chapter 8: Sorting & ranking

### Sorting values

Sorting your DataFrame or Series is also a useful and common operation. This is as easy as providing a column to sort on, and *Pandas* will do the rest. By default this will sort the rows of the DataFrame in ascending order (lowest to highest), but you can change this by setting the optional `ascending` parameter to `False`. Note that strings are automatically sorted in alphabetical order too.

The code below sorts the tracks by `artist` in ascending order:

In [ ]:
display(clean.sort_values(by="artist").head())

### Exercise 18

Sort `clean` by `streams`, highest first, and store the result in `by_streams`. Then show the top five tracks.

In [ ]:
by_streams = None

# YOUR CODE HERE

display(by_streams[["artist", "genre", "streams", "avg_rating"]].head())

In [ ]:
test_18(by_streams)

Another useful method is sorting by multiple columns: simply replace the `by=...` argument with a list of column names (`by=['col1', 'col2', ...]`). Similarly, you can replace the `ascending=...` argument with a list with `True` or `False` values (`ascending=[True, False, ...]`) of the same length, which indictates for each of the corresponding columns whether they should be sorted in ascending order or not.

Note that the order of the columns in the `by` list matters, as the DataFrame is sorted by the first column first, and so on. The code below first sorts by `genre` (ascending). Within each genre, it then sorts by `streams` (descending):

In [ ]:
display(
    clean.sort_values(by=["genre", "streams"], ascending=[True, False])
         [["genre", "artist", "streams"]]
         .head(8)
)

### Just the extremes: `nlargest` and `nsmallest`

If you only want the top or bottom few, sorting the whole frame is wasteful. `nlargest(n, column)` and `nsmallest(n, column)` give you exactly those rows.

> There is also `rank()`, which replaces each value by its position in the order (useful when you want a "rank" column rather than a reordered frame).

In [ ]:
print("most streamed:")
display(clean.nlargest(5, "streams")[["artist", "genre", "streams"]])

print("least streamed:")
display(clean.nsmallest(5, "streams")[["artist", "genre", "streams"]])

### Validation

Sorting can be a great way to check for outliers. Or even find values that should not exist a negative price, a date in 1900, a rating of 47.

Here the extremes are large but believable: the top track has 7.9 million streams and the smallest is 501. These are plausible values. Compare that with what the same check would have found before the repair: the maximum was 97 365, because everything larger had been silently deleted.

In [ ]:

sorted1 = clean.sort_values(by="streams")
print("fewest streams:")
display(sorted1.head(1))
print("most streams:")
display(sorted1.tail(1))

## Chapter 9: Grouping & aggregation

Often we want to have information about groups. So, not "what is the average rating" but "what is the average rating per genre".

### Grouping

Categorical variables (such as the `genre` column in our dataset) can also be used to group data together. In doing this, you combine one or more rows from your original DataFrame into a row in the grouped DataFrame. You will then need to specify how these rows get combined (also called 'aggregating'), as multiple rows will need to be reduced to one row per Category.

First, you can group a DataFrame using the `pd.DataFrame.groupby` method and specifying what column you want to group on. This gives something called a `DataFrameGroupBy`, which is special intermediate `groupby` object. The groupby can then be completed by using an aggregate function on the `DataFrameGroupBy`. This will return a regular `DataFrame` again, with its rows reduced as specified by the aggregate function.

In [ ]:
grouped = clean.groupby("genre")

print("The intermediate groupby object, which isn't a DataFrame yet:")
display(grouped)

print("\nBut, it contains a DataFrame per group:")
for genre, group in grouped:
    print(f"Genre {genre}:")
    display(group.head(3))

You've already seen some of the possible aggregate functions for numerical values throughout this notebook. These are functions like `.sum()`, `.max()`, `.min()`, `.mean()` and `.median()`, which combine values by computing, for example, the maximum value in each column. With a `groupby`, these functions are applied to *all the remaining columns*, so the columns that were not selected as the column to group on. Applying an aggregate function to a `DataFrameGroupBy` results in a new DataFrame with just one row for each separate group, containing all the combined values of that group. Note that is not possible to apply these functions on *nonnumerical* columns (like the `artist` column), so trying this will result in an error. You can pass the optional `numeric_only=True` argument to the aggregate functions to tell *Pandas* explicitly to just drop those columns.

In [ ]:
print("mean per genre:")
display(grouped.mean(numeric_only=True))

print("maximum per genre:")
display(grouped.max(numeric_only=True))

As you can see, the result of these groupby and aggregates is a new DataFrame containing the combined data. In the mean DataFrame you can read off that a Pop track averages far more streams than a Classical one, and in the max DataFrame you can see the single most-streamed track within each genre.

There are also some aggregate functions that work on both numerical and categorical columns:

- `grouped_df.count()` will count the number of rows in that group, excluding `NaN`s.
- `grouped_df.nunique()` counts the number of unique values in each group for each column.
- `grouped_df.first()` gives the elements of the first of the grouped rows. Similarly, `.last()` gives the last.
- `grouped_df.head(n)` will include (up to) the first `n` rows in that group. Similarly, `.tail(n)` gives the last `n` rows.


In [ ]:
print("number of unique values in each group:")
display(clean.groupby("genre").nunique())

print("the first row of each group:")
display(clean.groupby("genre").first())

Like regular `DataFrames`, you can select one or more columns from a `DataFrameGroupBy` using the column names. Remember that, as when indexing columns, a single column will always return a Series, and a list of columns (even a list of 1 column) returns a DataFrame!

The examples below compute the mean number of streams per genre, and the number of unique values in `artist` and `year` per genre.

In [ ]:
grouped = clean.groupby("genre")

print("mean streams per genre (a Series, one column selected):")
display(grouped["streams"].mean())

print("unique artists and years per genre (a DataFrame, a list of columns):")
display(grouped[["artist", "year"]].nunique())

### Exercise 19

Compute the mean of `streams` and `avg_rating` for each genre, and store it in `mean_per_genre`.

Select the columns *before* aggregating, so that you don't need `numeric_only=True`.

In [ ]:
mean_per_genre = None

# YOUR CODE HERE

display(mean_per_genre)

In [ ]:
test_19(mean_per_genre)

Pandas supports many more methods of aggregating data, which you can use through the `grouped_df.agg()` method. Rather than 'just' computing the mean, you can give a list of methods to aggregate the groups in your DataFrame:

In [ ]:
display(clean.groupby("genre")[["streams", "avg_rating"]].agg(["count", "mean"]))

### Exercise 20

Create a DataFrame called `stream_stats` holding, for each genre, the number of tracks together with the minimum, mean and maximum number of streams.

In [ ]:
stream_stats = None

# YOUR CODE HERE

display(stream_stats)

In [ ]:
test_20(stream_stats)

### Validation

A `groupby` splits your rows into groups. So the sizes of those groups should add back up to the number of rows you started with. Check it:

In [ ]:
sizes = clean.groupby("genre").size()

display(sizes)
print("rows in clean:      ", len(clean))
print("sum of group sizes: ", sizes.sum())

So after grouping there are 440 rows of the 480 left. Forty rows are simply not there. They are the tracks with no `genre`. By default `groupby` drops rows whose grouping key is missing (silently, with no warning and no error). Every per-genre number you computed above, including your own answers to Exercises 20 and 21, quietly excludes 8% of the data.

Which is often what you want: there is no sensible genre to file those tracks under. The problem is not the behaviour, it is that nothing tells you it happened. If you had gone straight from `groupby` to a chart, the chart would have looked perfectly fine.

If you want to see them, pass `dropna=False`:

In [ ]:
all_sizes = clean.groupby("genre", dropna=False).size()

display(all_sizes)
print("sum of group sizes: ", all_sizes.sum())

Now it reconciles: 480.

This check is worth running after any operation that splits, joins or reshapes data. It costs one line and catches a lot of silent errors.

### Filling from a group

In Chapter 6 we filled the missing ratings with the overall average, and noted that this ignores something obvious: genres aren't rated alike.

Now we can check that claim:

In [ ]:
display(clean.groupby("genre")["avg_rating"].mean().round(2))

Classical averages around 4.3 and Pop around 3.4 (nearly a full point apart). Filling a missing Classical rating with the overall average of 3.8 understates it.

`transform` applies a group-wise calculation but hands back a Series *aligned to the original rows*, which is exactly what `fillna` needs:

In [ ]:
# start again from the frame where avg_rating was still missing
rated = tracks.dropna(subset=["streams"]).copy()

overall_fill = rated["avg_rating"].fillna(rated["avg_rating"].mean())
genre_fill = rated["avg_rating"].fillna(
    rated.groupby("genre")["avg_rating"].transform("mean")
)

differs = (genre_fill - overall_fill).abs() > 0.01
print("rows where the two fills disagree:", differs.sum())
print("largest difference:", round((genre_fill - overall_fill).abs().max(), 2))

Eleven of the 23 filled values change, by up to 0.41 — small, but systematic: it makes Classical tracks look worse and Pop tracks better than they are.

Note the same trap as before, from the other side: a track with no genre has no group to take a mean from, so `transform` gives it `NaN` and the fill leaves it missing. Whichever way you turn, the rows with a missing key need a decision of their own.

*Further Reading:* On [grouping data](https://jakevdp.github.io/PythonDataScienceHandbook/03.08-aggregation-and-grouping.html).